In [3]:
!pip install -q --upgrade --force-reinstall torch==2.10.0+cu128 torchvision==0.25.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers datasets peft trl accelerate
!pip install -q "torchao==0.10.0" "peft==0.14.0"
!git clone https://github.com/ggerganov/llama.cpp 2>/dev/null || true
!pip install -q -r llama.cpp/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 97.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 167.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 43.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 50.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 66.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 120.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 60.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 68.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 64.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 61.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 97.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
import sys
import torch
import subprocess

from google.colab import drive
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ======================
# -- PATHS
# ======================

MODEL_ID   = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATASET_ID = "g0ofycatz/OkinDataset"

BASE_DIR   = "/content/drive/MyDrive/okin"
OUTPUT_DIR = os.path.join(BASE_DIR, "okin_llm")
GGUF_DIR   = os.path.join(BASE_DIR, "gguf")
LLAMA_CPP  = "/content/llama.cpp"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

# ======================
# -- DATA
# ======================

dataset   = load_dataset(DATASET_ID, split="train")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# ======================
# -- MODEL
# ======================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

# ======================
# -- TRAIN
# ======================

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_pin_memory=True,
    auto_find_batch_size=True,
    assistant_only_loss=True,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_length=512,
    eos_token="<|im_end|>",
    optim="adamw_torch_fused",
)

trainer = SFTTrainer(
    model=model,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=args,
    train_dataset=dataset,
)

trainer.train()

merged = trainer.model.merge_and_unload()
merged.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ======================
# -- GGUF
# ======================

convert   = os.path.join(LLAMA_CPP, "convert_hf_to_gguf.py")
gguf_fp16 = os.path.join(GGUF_DIR, "okin-qwen-fp16.gguf")

print("Converting to GGUF...")
subprocess.run(
    [sys.executable, convert, OUTPUT_DIR, "--outfile", gguf_fp16, "--outtype", "f16"],
    check=True,
)

print(f"Done: {gguf_fp16}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,4.324538
20,3.619540
30,3.084649


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]